# 02 - Compreensão dos Dados

**Projeto:** Análise do Mercado Imobiliário Português  
**Fase CRISP-DM:** Data Understanding  
**Objetivo:** compreender a estrutura, qualidade, distribuições e padrões iniciais do dataset antes da preparação avançada e da modelação.  
**Última alteração:** 19/06/2026

Este notebook analisa o dataset imobiliário português com foco em qualidade dos dados, estatísticas descritivas, distribuição da variável alvo e relações exploratórias entre localização, tipologia, áreas e preço.

Entradas esperadas:
- Dataset processado inicial, quando disponível.
- Dataset raw local, no Kaggle ou no GitHub, como alternativa para garantir reprodutibilidade.

Saídas esperadas:
- Diagnóstico inicial da qualidade dos dados.
- Tabelas e gráficos exploratórios.
- Observações fundamentadas para orientar limpeza, feature engineering e modelação.

Nota metodológica:
- Esta fase identifica padrões e hipóteses, mas não estabelece causalidade. Qualquer conclusão substantiva deve ser confirmada nas fases estatística e de modelação.


## Perguntas de Análise

- Qual é a dimensão e composição do dataset após a preparação inicial?
- Que variáveis têm maior proporção de valores em falta?
- Existem duplicados, valores inválidos ou valores extremos que precisem de análise específica?
- Como se distribuem os anúncios por localização e tipo de imóvel?
- Que variáveis parecem exigir tratamento antes da modelação?


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
GITHUB_BASE_URL = "https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main"
DATASET_FILENAME = "portugal_listinigs.csv"
PROCESSED_FILENAME = "portugal_listings_initial_clean.csv"
GITHUB_RAW_DATA_URL = f"{GITHUB_BASE_URL}/data/raw/{DATASET_FILENAME}"
GITHUB_PROCESSED_DATA_URL = f"{GITHUB_BASE_URL}/data/processed/{PROCESSED_FILENAME}"


def find_project_root(start=None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        markers = [candidate / "README.md", candidate / "requirements.txt", candidate / "data"]
        if all(marker.exists() for marker in markers):
            return candidate
    return Path("/kaggle/working") if Path("/kaggle/working").exists() else current


def find_raw_data_source(project_root: Path):
    local_candidates = [
        project_root / "data" / "raw" / DATASET_FILENAME,
        (Path.cwd() / ".." / "data" / "raw" / DATASET_FILENAME).resolve(),
    ]

    for path in local_candidates:
        if path.exists():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(DATASET_FILENAME))
        if matches:
            return matches[0]

    return GITHUB_RAW_DATA_URL


def processed_data_path(project_root: Path) -> Path:
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working") / PROCESSED_FILENAME
    return project_root / "data" / "processed" / PROCESSED_FILENAME


def figures_dir(project_root: Path) -> Path:
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working") / "figures"
    return project_root / "reports" / "figures"


PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = find_raw_data_source(PROJECT_ROOT)
PROCESSED_DATA_PATH = processed_data_path(PROJECT_ROOT)
FIGURES_DIR = figures_dir(PROJECT_ROOT)

print(f"Raiz de trabalho: {PROJECT_ROOT}")
print(f"Dataset original: {RAW_DATA_PATH}")
print(f"Dataset processado: {PROCESSED_DATA_PATH}")


Raiz de trabalho: /kaggle/working
Dataset original: https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main/data/raw/portugal_listinigs.csv
Dataset processado: /kaggle/working/portugal_listings_initial_clean.csv


## Carregamento

O notebook usa a versão processada criada na inicialização quando ela existe. Caso contrário, carrega o ficheiro original para permitir uma primeira inspeção sem alterar `data/raw/`.


In [3]:
def read_first_available_csv(sources):
    errors = []
    for source in sources:
        if isinstance(source, Path) and not source.exists():
            continue
        try:
            return pd.read_csv(source, low_memory=False), source
        except Exception as exc:
            errors.append(f"{source}: {exc}")
    raise FileNotFoundError("Não foi possível carregar nenhum CSV. " + " | ".join(errors))


processed_sources = [PROCESSED_DATA_PATH, GITHUB_PROCESSED_DATA_URL]
raw_sources = [RAW_DATA_PATH, GITHUB_RAW_DATA_URL]

df, DATA_SOURCE_USED = read_first_available_csv([*processed_sources, *raw_sources])

column_names = {
    "Price": "price",
    "District": "district",
    "City": "city",
    "Town": "town",
    "Type": "type",
    "EnergyCertificate": "energy_certificate",
    "GrossArea": "gross_area",
    "TotalArea": "total_area",
    "Parking": "parking",
    "HasParking": "has_parking",
    "Floor": "floor",
    "ConstructionYear": "construction_year",
    "EnergyEfficiencyLevel": "energy_efficiency_level",
    "PublishDate": "publish_date",
    "Garage": "garage",
    "Elevator": "elevator",
    "ElectricCarsCharging": "electric_cars_charging",
    "TotalRooms": "total_rooms",
    "NumberOfBedrooms": "number_of_bedrooms",
    "NumberOfWC": "number_of_wc",
    "ConservationStatus": "conservation_status",
    "LivingArea": "living_area",
    "LotSize": "lot_size",
    "BuiltArea": "built_area",
    "NumberOfBathrooms": "number_of_bathrooms",
}

df = df.rename(columns=column_names)

for col in [
    "price",
    "gross_area",
    "total_area",
    "living_area",
    "lot_size",
    "built_area",
    "construction_year",
    "total_rooms",
    "number_of_bedrooms",
    "number_of_wc",
    "number_of_bathrooms",
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "publish_date" in df.columns:
    df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce")

print(f"Fonte usada: {DATA_SOURCE_USED}")
print(f"Dataset carregado: {df.shape[0]:,} linhas e {df.shape[1]} colunas")
display(df.head())


Fonte usada: https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main/data/processed/portugal_listings_initial_clean.csv
Dataset carregado: 126,242 linhas e 29 colunas


,price,district,city,town,type,energy_certificate,gross_area,total_area,parking,has_parking,...,number_of_wc,conservation_status,living_area,lot_size,built_area,number_of_bathrooms,price_m2,property_age,publish_year,publish_month
0,780000.0,Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.0,552450.0,0.0,False,...,NaN,NaN,120.0,NaN,NaN,0.0,1.411892,NaN,NaN,NaN
1,223000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.0,1.0,True,...,NaN,NaN,81.0,NaN,NaN,2.0,2753.086420,NaN,NaN,NaN
2,228000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.0,1.0,True,...,NaN,NaN,108.0,NaN,NaN,2.0,2111.111111,NaN,NaN,NaN
3,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,NaN,NaN,114.0,NaN,NaN,0.0,2192.982456,NaN,NaN,NaN
4,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,NaN,NaN,114.0,NaN,NaN,2.0,2192.982456,NaN,NaN,NaN


### Leitura dos resultados

O dataset foi carregado com sucesso a partir da primeira fonte disponível. A amostra confirma que a base já contém colunas normalizadas e variáveis derivadas iniciais, permitindo avançar para análise de estrutura, missing values e distribuição do preço.


## Estrutura Geral

Esta secção prepara métricas básicas sobre dimensão, colunas e tipos. As observações devem ser escritas depois de executar e rever os resultados.


In [4]:
dataset_shape = {
    "linhas": df.shape[0],
    "colunas": df.shape[1],
}

columns_overview = pd.DataFrame({
    "coluna": df.columns,
    "tipo": df.dtypes.astype(str).values,
})

display(pd.Series(dataset_shape).to_frame("valor"))
display(columns_overview)


,valor
linhas,126242
colunas,29


,coluna,tipo
0,price,float64
1,district,object
2,city,object
3,town,object
4,type,object
5,energy_certificate,object
6,gross_area,float64
7,total_area,float64
8,parking,float64
9,has_parking,object


### Leitura dos resultados

A estrutura mostra a dimensão da base e os tipos de dados disponíveis. Esta visão é importante para separar variáveis numéricas, categóricas e temporais antes de qualquer análise estatística ou modelação.


## Valores em Falta

A análise de valores em falta deve distinguir ausência esperada por tipo de imóvel de falhas de recolha ou campos críticos para modelação.


In [5]:
missing_summary = (
    pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "valores_em_falta": df.isna().sum().values,
        "percentagem_em_falta": (df.isna().mean().values * 100).round(2),
        "valores_unicos": df.nunique(dropna=True).values,
    })
    .sort_values("percentagem_em_falta", ascending=False)
    .reset_index(drop=True)
)

display(missing_summary.head(20))


,coluna,tipo,valores_em_falta,percentagem_em_falta,valores_unicos
0,conservation_status,object,108263,85.76,6
1,built_area,float64,101711,80.57,7290
2,gross_area,float64,100574,79.67,2264
3,floor,object,100224,79.39,19
4,publish_month,float64,98770,78.24,12
5,publish_date,datetime64[ns],98770,78.24,27397
6,publish_year,float64,98770,78.24,7
7,lot_size,float64,90113,71.38,6770
8,number_of_bedrooms,float64,82586,65.42,22
9,number_of_wc,float64,73208,57.99,26


### Leitura dos resultados

Os valores em falta não estão distribuídos de forma uniforme. Algumas colunas têm ausência elevada e podem representar informação estrutural, por exemplo áreas ou atributos que não se aplicam a certos tipos de imóvel. Por isso, a imputação deve ser decidida por variável e contexto.


## Duplicados

Duplicados exatos podem representar anúncios repetidos ou artefactos de recolha. A decisão de remover deve ficar documentada na preparação dos dados.


In [6]:
duplicate_summary = {
    "duplicados_exatos": int(df.duplicated().sum()),
    "percentagem_duplicados": round(float(df.duplicated().mean() * 100), 2),
}

display(pd.Series(duplicate_summary).to_frame("valor"))


,valor
duplicados_exatos,0.0
percentagem_duplicados,0.0


### Leitura dos resultados

A verificação de duplicados confirma se a versão analisada já passou por remoção de duplicados exatos. Este controlo evita que anúncios repetidos influenciem distribuições, contagens e futuras métricas de modelo.


## Variáveis Numéricas

Esta secção cria resumos para apoiar a identificação de escalas, possíveis valores inválidos e valores extremos. A presença de valores extremos deve ser interpretada por contexto, localização e tipo de imóvel.


In [7]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

numeric_summary = df[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

display(numeric_summary)


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
price,126242.0,370785.559296,3.935295e+06,1.000000,5999.4100,20000.000000,85000.000000,210000.000000,390000.000000,1.198947e+06,2.750000e+06,1.380000e+09
gross_area,25668.0,2967.226586,1.197076e+05,1.000000,25.0000,51.000000,102.000000,165.000000,300.000000,2.381650e+03,1.960694e+04,1.275000e+07
total_area,117571.0,555690.020881,1.791771e+08,1.000000,24.0000,47.000000,92.000000,160.000000,520.000000,8.854500e+03,5.152000e+04,6.142007e+10
parking,126096.0,0.544284,8.389423e-01,0.000000,0.0000,0.000000,0.000000,0.000000,1.000000,3.000000e+00,3.000000e+00,3.000000e+00
construction_year,83118.0,1988.770146,2.658058e+01,1900.000000,1935.0000,1937.000000,1972.000000,1993.000000,2008.000000,2.024000e+03,2.024000e+03,2.025000e+03
total_rooms,68605.0,3.235610,1.082976e+01,0.000000,0.0000,0.000000,2.000000,3.000000,4.000000,7.000000e+00,1.200000e+01,2.751000e+03
number_of_bedrooms,43656.0,2.672851,1.794427e+00,0.000000,0.0000,0.000000,2.000000,3.000000,3.000000,5.000000e+00,9.000000e+00,2.100000e+01
number_of_wc,53034.0,0.433665,1.015513e+00,0.000000,0.0000,0.000000,0.000000,0.000000,1.000000,2.000000e+00,4.000000e+00,5.900000e+01
living_area,97426.0,1430.438928,3.574641e+04,1.000000,26.0000,43.000000,80.000000,118.000000,206.000000,1.800000e+03,1.920000e+04,5.429000e+06
lot_size,36129.0,58166.804091,5.426151e+06,1.000000,36.0000,85.000000,276.000000,720.000000,3000.000000,2.210540e+04,1.338486e+05,9.923010e+08


### Leitura dos resultados

As estatísticas numéricas indicam assimetria forte e valores extremos em preço, áreas e algumas contagens. Isto sugere que médias simples podem ser enganadoras e que percentis, medianas e transformações logarítmicas serão mais informativos.


In [8]:
invalid_rules = {}

for col in ["price", "gross_area", "total_area", "living_area", "lot_size", "built_area"]:
    if col in df.columns:
        invalid_rules[f"{col}_menor_ou_igual_zero"] = int((df[col] <= 0).sum())

for col in ["total_rooms", "number_of_bedrooms", "number_of_wc", "number_of_bathrooms"]:
    if col in df.columns:
        invalid_rules[f"{col}_negativo"] = int((df[col] < 0).sum())

if "construction_year" in df.columns:
    invalid_rules["construction_year_menor_1900"] = int((df["construction_year"] < 1900).sum())
    invalid_rules["construction_year_maior_2026"] = int((df["construction_year"] > 2026).sum())

invalid_values_summary = pd.DataFrame(
    invalid_rules.items(),
    columns=["regra", "registos"],
)

display(invalid_values_summary.sort_values("registos", ascending=False))


,regra,registos
0,price_menor_ou_igual_zero,0
1,gross_area_menor_ou_igual_zero,0
2,total_area_menor_ou_igual_zero,0
3,living_area_menor_ou_igual_zero,0
4,lot_size_menor_ou_igual_zero,0
5,built_area_menor_ou_igual_zero,0
6,total_rooms_negativo,0
7,number_of_bedrooms_negativo,0
8,number_of_wc_negativo,0
9,number_of_bathrooms_negativo,0


### Leitura dos resultados

As regras de invalidez ajudam a distinguir problemas objetivos de domínio de valores apenas extremos. Valores não positivos em áreas ou preços são incoerentes; valores muito altos devem ser analisados antes de qualquer exclusão.


## Variáveis Categóricas

A análise categórica deve observar cardinalidade, categorias raras e concentração por localização ou tipo de imóvel.


In [9]:
categorical_cols = df.select_dtypes(include=["object", "category", "boolean"]).columns.tolist()

categorical_summary = (
    pd.DataFrame({
        "coluna": categorical_cols,
        "valores_unicos": [df[col].nunique(dropna=True) for col in categorical_cols],
        "percentagem_em_falta": [round(float(df[col].isna().mean() * 100), 2) for col in categorical_cols],
    })
    .sort_values("valores_unicos", ascending=False)
    .reset_index(drop=True)
)

display(categorical_summary)


,coluna,valores_unicos,percentagem_em_falta
0,town,2263,0.00
1,city,275,0.00
2,district,27,0.00
3,type,21,0.01
4,floor,19,79.39
5,energy_certificate,12,0.01
6,energy_efficiency_level,11,50.52
7,conservation_status,6,85.76
8,has_parking,2,49.50
9,garage,2,50.52


### Leitura dos resultados

As variáveis categóricas têm cardinalidades muito diferentes. Colunas como `district` e `type` são diretamente úteis para análise, enquanto `city`, `town` e combinações geográficas poderão exigir tratamento de categorias raras antes da modelação.


In [10]:
key_frequency_tables = {}

for col in ["district", "city", "town", "type", "energy_certificate"]:
    if col in df.columns:
        key_frequency_tables[col] = (
            df[col]
            .value_counts(dropna=False)
            .head(15)
            .rename_axis(col)
            .reset_index(name="registos")
        )

for col, table in key_frequency_tables.items():
    print(f"Frequências principais: {col}")
    display(table)


Frequências principais: district


,district,registos
0,Lisboa,29883
1,Porto,21114
2,Setúbal,10817
3,Braga,10551
4,Faro,8010
5,Coimbra,7299
6,Aveiro,6832
7,Santarém,6727
8,Leiria,6637
9,Castelo Branco,4726


Frequências principais: city


,city,registos
0,Lisboa,8041
1,Sintra,5311
2,Porto,4887
3,Vila Nova de Gaia,3952
4,Cascais,3079
5,Braga,2340
6,Coimbra,2105
7,Oeiras,2090
8,Loures,1999
9,Guimarães,1861


Frequências principais: town


,town,registos
0,Paranhos,1880
1,Cascais e Estoril,1417
2,Albufeira e Olhos de Água,1203
3,Portimão,1004
4,"Cedofeita, Santo Ildefonso, Sé, Miragaia, São ...",999
5,"Algés, Linda-a-Velha e Cruz Quebrada-Dafundo",795
6,Quarteira,790
7,Canidelo,753
8,Arroios,751
9,Montijo e Afonsoeiro,704


Frequências principais: type


,type,registos
0,Apartment,43846
1,House,34445
2,Land,29458
3,Store,5055
4,Farm,3755
5,Building,2392
6,Transfer of lease,1579
7,Warehouse,1338
8,Garage,885
9,Other - Commercial,762


Frequências principais: energy_certificate


,energy_certificate,registos
0,NC,56300
1,C,15745
2,D,15042
3,E,10527
4,A,8801
5,F,7231
6,B-,4260
7,B,4051
8,A+,3788
9,G,425


### Leitura dos resultados

As tabelas de frequência mostram concentração em algumas localizações e tipologias. Esta concentração é útil para análise, mas também alerta para possíveis desequilíbrios que podem afetar comparações entre grupos e modelos preditivos.


## Distribuição da Variável Alvo

A variável `price` deve ser analisada com cuidado, sobretudo por causa de valores extremos. Qualquer transformação ou remoção deve ser justificada antes da modelação.


In [11]:
if "price" in df.columns:
    target_summary = df["price"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
else:
    target_summary = pd.Series(dtype="float64")

display(target_summary.to_frame("price"))


,price
count,1.262420e+05
mean,3.707856e+05
std,3.935295e+06
min,1.000000e+00
1%,5.999410e+03
5%,2.000000e+04
25%,8.500000e+04
50%,2.100000e+05
75%,3.900000e+05
95%,1.198947e+06


### Leitura dos resultados

A variável `price` apresenta forte assimetria à direita, com diferença relevante entre mediana, percentis superiores e máximo. Isto reforça a necessidade de usar métricas robustas e avaliar `log_price` em fases posteriores.


## Análise por Localização e Tipo

Esta secção prepara agregações base para comparar volume de anúncios e preço anunciado por grupos relevantes. As comparações devem evitar conclusões causais.


In [12]:
group_summaries = {}

for col in ["district", "city", "type"]:
    if {col, "price"}.issubset(df.columns):
        group_summaries[col] = (
            df.groupby(col, observed=True)["price"]
            .agg(registos="count", mediana="median", media="mean")
            .sort_values("registos", ascending=False)
            .reset_index()
        )

for col, table in group_summaries.items():
    print(f"Resumo de preço por {col}")
    display(table.head(15))


Resumo de preço por district


,district,registos,mediana,media
0,Lisboa,29883,349000.0,575163.637995
1,Porto,21114,257520.0,346598.287913
2,Setúbal,10817,273000.0,424677.537672
3,Braga,10551,185000.0,248520.094304
4,Faro,8010,295000.0,704239.060134
5,Coimbra,7299,110000.0,186136.119057
6,Aveiro,6832,168975.0,254267.022395
7,Santarém,6727,95000.0,189939.073703
8,Leiria,6637,140000.0,227792.665361
9,Castelo Branco,4726,45000.0,99313.959585


Resumo de preço por city


,city,registos,mediana,media
0,Lisboa,8041,525000.0,7.565496e+05
1,Sintra,5311,225000.0,4.181556e+05
2,Porto,4887,345000.0,4.682301e+05
3,Vila Nova de Gaia,3952,292125.0,4.098278e+05
4,Cascais,3079,750000.0,1.164424e+06
5,Braga,2340,241500.0,3.025781e+05
6,Coimbra,2105,180000.0,2.703271e+05
7,Oeiras,2090,499500.0,6.441488e+05
8,Loures,1999,290000.0,3.836553e+05
9,Guimarães,1861,196000.0,2.661249e+05


Resumo de preço por type


,type,registos,mediana,media
0,Apartment,43846,279000.0,3.731907e+05
1,House,34445,250000.0,4.552882e+05
2,Land,29458,65000.0,1.962310e+05
3,Store,5055,130000.0,2.404414e+05
4,Farm,3755,260000.0,5.932844e+05
5,Building,2392,550000.0,8.541717e+05
6,Transfer of lease,1579,69000.0,1.285224e+05
7,Warehouse,1338,277500.0,6.090510e+05
8,Garage,885,23500.0,4.015873e+04
9,Other - Commercial,762,230000.0,4.940573e+05


### Leitura dos resultados

Os resumos por localização e tipo sugerem diferenças claras na distribuição de preços. Estas diferenças são descritivas e não causais: podem refletir área, segmento, qualidade do anúncio, localização específica e outros fatores não controlados.


## Observações Intermédias

A leitura exploratória deve ser interpretada como diagnóstico inicial. Nesta fase, os aspetos mais importantes a acompanhar são a assimetria da variável `price`, a existência de valores em falta em variáveis descritivas, a coerência das áreas e a variação do preço por localização e tipo de imóvel.

As observações finais devem ser revistas sempre que o dataset for atualizado ou quando novas regras de limpeza forem aplicadas.


## Conclusão

Este notebook organiza a fase de compreensão dos dados e cria uma primeira leitura sobre qualidade, estrutura e comportamento das principais variáveis do mercado imobiliário português. A análise deve servir como ponte entre a limpeza inicial e as decisões seguintes de preparação, engenharia de variáveis e modelação.

Principais pontos a reter:
- A variável `price` deve ser tratada com cuidado, porque preços imobiliários tendem a apresentar assimetria e valores extremos.
- Valores em falta devem ser analisados por coluna e por contexto antes de qualquer imputação.
- Comparações por localização e tipo de imóvel são essenciais, mas não devem ser interpretadas como efeitos causais nesta fase.
- A preparação seguinte deve preservar rastreabilidade entre dados raw, dados processados e decisões analíticas.

Próxima etapa:
- Documentar regras de qualidade e avançar para preparação de dados, feature engineering e validação estatística das principais relações observadas.

**Última alteração:** 19/06/2026
